# Inspect saved VWAPSb curves

Read the saved output without recalculating prices. Select an exact product/region/unit identity, a date and all contract types together or one family. The second tab compares means over a date range for **fixed delivery intervals**, using the same paired dates for output and EEX. Relative tenor rolls therefore do not mix different physical contracts.

Run `python run.py demo` first for invented example data. Select this project's Python environment in Jupyter or VS Code. Code and controls are in English.


In [ ]:
PROJECT_ROOT = None  # Example: r"C:\Projects\VWAPSb".
CONFIG_PATH = None  # Prefer output/demo/config.toml when available, otherwise config.toml.
OUTPUT_PATH = None  # Default: the selected config's output_dir / "filled.csv".
# To inspect a backtest candidate, use its scores.csv instead.
# OUTPUT_PATH = "output/demo/results/backtest/scores.csv"


In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
from datetime import date

import pandas as pd
from IPython.display import display

if PROJECT_ROOT is None:
    current = Path.cwd().resolve()
    ROOT = next((path for path in [current, *current.parents] if (path / "src/vwapsb").is_dir()), None)
    if ROOT is None:
        raise ValueError("Set PROJECT_ROOT to the VWAPSb repository directory.")
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if not (ROOT / "src/vwapsb").is_dir():
    raise ValueError("PROJECT_ROOT must contain src/vwapsb.")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None

def project_path(value):
    """Resolve notebook paths relative to the repository root."""
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()

from vwapsb.config import load_config
from vwapsb.viewer import create_viewer, load_output

config_path = project_path(CONFIG_PATH) if CONFIG_PATH is not None else (
    ROOT / "output/demo/config.toml" if (ROOT / "output/demo/config.toml").is_file() else ROOT / "config.toml"
)
config = load_config(config_path)
output_path = project_path(OUTPUT_PATH) if OUTPUT_PATH is not None else config.resolve(config.output_dir) / "filled.csv"
data = load_output(output_path)
print(f"Loaded {len(data):,} saved rows from {output_path}")


In [ ]:
viewer = create_viewer(data)
display(viewer)


The viewer only shows rows present in the file. It does not manufacture missing contracts, adjust prices or load current EEX files. EEX values are the references saved alongside the output. For backtest `scores.csv`, the chart shows withheld-observation predictions and the hidden actual values; these are not complete refilled curves. Missing predictions remain important even when a line cannot be drawn.
